In [ ]:
from io import BytesIO
from pathlib import Path
from openpyxl import Workbook, load_workbook

In [ ]:

source_path = Path("SI-dataset.xlsx")  # Path to the original SI workbook.
output_path = Path("derived/intermediate") / "dataset.xlsx"  # Path to the generated workbook.

In [ ]:
INPUT_COLUMNS = (      ("Catalyst number", "Catalyst Number"),  # Input columns.
    ("Reference number", "Reference Number"),
    ("Catalyst Name", "Catalyst Name"),
    ("Main Phase", "Phase"),
    ("wt% Cu", "Cu Loading [wt.%]"),
    ("Support1", "Support 1"),
    ("χ Support1", "Electronegativity Support 1 [/]"),
    ("wt% Support1", "Support 1 Loading [wt.%]"),
    ("Support2", "Support 2"),
    ("χ Support2", "Electronegativity Support 2 [/]"),
    ("wt% Support2", "Support 2 Loading [wt.%]"),
    ("Support3", "Support 3"),
    ("χ Support3", "Electronegativity Support 3 [/]"),
    ("wt% Support3", "Support 3 Loading [wt.%]"),
    ("Promoter", "Promoter"),
    ("χ Promoter", "Electronegativity Promoter"),
    ("wt% Promoter", "Promoter Loading [wt.%]"),
    ("Calc T", "Calcination Temperature [K]"),
    ("Calc D", "Calcination Duration [h]"),
    ("Method", "Method"),
    ("Synthesis Method", "Synthesis Method"),
    ("GHSV", "GHSV [mL·h-1·gcat-1]"),
    ("H2/CO2", "H2/CO2 [-]"),
    ("P", "Pressure [MPa]"),
    ("T", "Temperature [K]"),
    ("EqCon", "EqCon [%]"),
    ("dCu", "dCu [nm]"),
    ("SBET", "SBET [m2 g-1]"),
    ("SACu", "SACu [m2 g-1]"),
    ("DCu", "DCu [%]"),
    ("Con", "Con [%]"),
    ("Sel", "Sel [%]"),
    ("STY", "STY [mgMeOH h-1 gcat-1]"),
    ("Reference DOI", "Reference DOI"),
    ("Data point number", "Number"),
    ("Byproduct", "By Product"),
    ("Feed Composition", "Feed Composition"),
)

In [ ]:
def read_si_dataset(source):  # Map SI worksheet columns to rows for the modeling workbook.
    workbook = load_workbook(source, read_only=True, data_only=True)  # Open Excel workbook.
    try:
        if "data" not in workbook.sheetnames:
            raise ValueError("The SI workbook is missing the 'data' sheet.")

        rows = workbook["data"].iter_rows(values_only=True)  # Rows accumulated for the result table.
        header = next(rows, ())  # Header.
        header_positions = {}  # Header positions.
        for index, name in enumerate(header):  # Current position or index label; Current item name.
            key = str(name).strip().casefold()  
            header_positions.setdefault(key, []).append(index)

        positions = {}  
        for output_name, expected_header in INPUT_COLUMNS:  # Output name; Expected header.
            matches = header_positions.get(expected_header.casefold(), [])  
            if len(matches) != 1:
                raise ValueError(f"Expected one SI column named {expected_header!r}, found {len(matches)}.")
            positions[output_name] = matches[0]

        records = []  # Experimental records selected for processing.
        point_numbers = set()  
        for row_number, row in enumerate(rows, start=2):  
            if all(value is None for value in row): 
                continue
            record = {name: row[index] for name, index in positions.items()}  # Current experimental record. Current item name; Current position or index label.

            for name in ("Data point number", "Catalyst number", "Reference number"):
                if type(record[name]) is not int or record[name] <= 0:
                    raise ValueError(f"SI data row {row_number}: invalid {name}.")
            point_number = record["Data point number"]  
            if point_number in point_numbers:
                raise ValueError(f"SI data row {row_number}: duplicate point {point_number}.")
            point_numbers.add(point_number)

            for name in ("Calc T", "Calc D"):
                value = record[name]  
                if isinstance(value, str) and value.strip().casefold() == "not reported":
                    record[name] = None

            for name in ("Support1", "Support2", "Support3", "Promoter"):
                value = record[name]
                if isinstance(value, str):
                    value = value.strip()
                    record[name] = 0 if value in {"0", "0.00"} else value

            conversion = record["Con"]  
            equilibrium = record["EqCon"]  # Equilibrium conversion or gas composition for the current case.
            record["Con/EqCon"] = (
                conversion / equilibrium
                if conversion is not None and equilibrium is not None and equilibrium > 0
                else None
            )
            records.append(record)

        if not records:
            raise ValueError("The SI 'data' sheet contains no data rows.")
        return records
    finally:
        workbook.close()


def write_model_input(records, output):  # Write the extracted records and column names to an Excel workbook.
    workbook = Workbook(write_only=True)  # Open Excel workbook.
    sheet = workbook.create_sheet("data")  # Result workbook worksheet name.
    columns = [name for name, _ in INPUT_COLUMNS]  # Columns included in the operation. Current item name.
    columns.insert(columns.index("EqCon") + 1, "Con/EqCon")
    sheet.append(columns)
    for record in records:  # Current experimental record.
        sheet.append([record[name] for name in columns])  # Current item name.

    with BytesIO() as contents:  # In-memory Excel workbook bytes.
        workbook.save(contents)
        output.parent.mkdir(parents=True, exist_ok=True)
        with output.open("xb") as handle:  # Open file handle.
            handle.write(contents.getbuffer())

In [ ]:
output_path.parent.mkdir(parents=True, exist_ok=True)
if output_path.exists():
    raise FileExistsError(f"Choose a new output path: {output_path.resolve()}")

records = read_si_dataset(source_path)  # Experimental records selected for processing.
write_model_input(records, output_path)

print(f"Data points: {len(records)}")
print(f"Catalysts: {len({row['Catalyst number'] for row in records})}") 
print(f"References: {len({row['Reference number'] for row in records})}")  
print(f"Saved to: {output_path.resolve()}")